<a href="https://colab.research.google.com/github/norahmunoz/ds2002-fa26/blob/main/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [25]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [26]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [27]:
before = len(df)
df = df.drop_duplicates()
rows_affected = before - len(df)

log('drop duplicates', 'Dropped duplicate rows', rows_affected)

[drop duplicates] Dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [28]:
df['price'] = df['price'].str.replace('$', '', regex=False).astype(float)

log('clean price', 'Removed dollar signs and converted price to float', len(df))

[clean price] Removed dollar signs and converted price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [29]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

before = len(df)
df = df.dropna(subset=['qty'])
df = df[df['qty'] >= 1]
rows_affected = before - len(df)

log('clean qty', 'Dropped missing and negative quantities', rows_affected)

[clean qty] Dropped missing and negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [30]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

before = df['item'].nunique()
df['item'] = df['item'].map(ITEM_MAP)
after = df['item'].nunique()

log('canonicalize item', 'Collapsed spelling variants into three product names', before - after)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[canonicalize item] Collapsed spelling variants into three product names (3 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [31]:
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}

before = df['category'].nunique()
df['category'] = df['category'].map(CATEGORY_MAP)
after = df['category'].nunique()

log('normalize category', 'Combined Apparel with Merch and standardized category names', before - after)

[normalize category] Combined Apparel with Merch and standardized category names (3 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [32]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin(['Cheeseburger', 'Foam Finger', 'Rain Poncho']).all()
assert df['category'].isin(['Food', 'Merch', 'RainGear']).all()

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [33]:
df['revenue'] = df['qty'] * df['price']

revenue_by_category = (
    df.groupby('category')['revenue']
      .sum()
      .sort_values(ascending=False)
      .round(2)
)

print(revenue_by_category)
print('Overall total:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Overall total: 4740.0


**What I would tell the vendor: I would tell the vendor to stock more Food items because Food generated the highest revenue at $1,656.00.

### TODO 8 — read back your log

In [34]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop duplicates,Dropped duplicate rows,15
1,clean price,Removed dollar signs and converted price to float,300
2,clean qty,Dropped missing and negative quantities,25
3,canonicalize item,Collapsed spelling variants into three product...,3
4,normalize category,Combined Apparel with Merch and standardized c...,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)** The cleaning step that changed the revenue total the most was removing missing and negative quantities. Revenue was 4,594.50 dollars before this step and 4,740.00 dollars after it.

**b)** One reasonable alternative would have been to keep Apparel and Merch as separate categories instead of combining them. This would not have changed the overall reported revenue because it only changes how the rows are grouped. I combined them because both categories represent merchandise, which makes the category totals more useful for making stocking decisions.
